# Explore here

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

url = "https://breathecode.herokuapp.com/asset/internal-link?id=2325&path=adult-census-income.csv"
df = pd.read_csv(url)
df = df.replace(' ?', np.nan).dropna()
df['high_income'] = df['income'].apply(lambda x: 1 if '>50K' in str(x).strip() else 0)

features = ['age', 'education', 'marital.status', 'occupation', 'hours.per.week', 'sex', 'native.country']
X = df[features]
y = df['high_income']

numeric_features = ['age', 'hours.per.week']
categorical_features = ['education', 'marital.status', 'occupation', 'sex', 'native.country']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
    ]
)

model.fit(X, y)

def recommend_trajectory(user_profile):
    user_df = pd.DataFrame([user_profile])
    prediction = model.predict(user_df)[0]
    prob = model.predict_proba(user_df)[0][1]
    
    if prediction == 1:
        return f"✅ With this profile, your probability of earning more than 50K is {prob*100:.1f}%."
    else:
        return f"⚠️ With this profile, your probability of exceeding 50K is only {prob*100:.1f}%. Consider improving your education or changing your occupation."

profiles = [
    {
        "name": "Case 1: Entry-level part-time worker",
        "profile": {
            "age": 22,
            "education": "HS-grad",
            "marital.status": "Never-married",
            "occupation": "Handlers-cleaners",
            "hours.per.week": 20,
            "sex": "Male",
            "native.country": "United-States"
        }
    },
    {
        "name": "Case 2: Mid-career full-time worker",
        "profile": {
            "age": 35,
            "education": "Some-college",
            "marital.status": "Married-civ-spouse",
            "occupation": "Adm-clerical",
            "hours.per.week": 40,
            "sex": "Female",
            "native.country": "United-States"
        }
    },
    {
        "name": "Case 3: Experienced professional with advanced degree",
        "profile": {
            "age": 45,
            "education": "Doctorate",
            "marital.status": "Married-civ-spouse",
            "occupation": "Prof-specialty",
            "hours.per.week": 50,
            "sex": "Male",
            "native.country": "United-States"
        }
    }
]

for item in profiles:
    print(f"--- {item['name']} ---")
    result = recommend_trajectory(item['profile'])
    print(result)
    print("\n")

--- Case 1: Entry-level part-time worker ---
⚠️ With this profile, your probability of exceeding 50K is only 0.0%. Consider improving your education or changing your occupation.


--- Case 2: Mid-career full-time worker ---
⚠️ With this profile, your probability of exceeding 50K is only 19.5%. Consider improving your education or changing your occupation.


--- Case 3: Experienced professional with advanced degree ---
✅ With this profile, your probability of earning more than 50K is 99.5%.


